In [1]:
# Imports
import pandas as pd
import torch

In [2]:
# Loading dataset and looking at first five rows
delay_df = pd.read_excel('data/Fleet Dealy Data from P2301 to P2611.xlsx')

delay_df.head()

,Inc.#,Date,Status,Bugle Mgr,Bugle Mgr Title,Trust Mgr,NR Mgr,Cause,Root Cause,Root Cause Descrip.,...,Region.,Sect2,Day 8,NR KPI,TOC1,TOC2,Region,Funct.,Unnamed: 53,1246.5
0,480134220401,2022-04-01,A,MHB2,HITACHI - CLASS 801,MHB2,OQIE,MN,NaN,No Root Cause entered,...,Azuma,Hitachi - Class 801,2022-04-08,Fleet,HB,LNER,Azuma,Azuma,NaN,NaN
1,485541220403,2022-04-03,A,MHBK,YARD - CRAIGENTINNY,MHBK,OQL9,MU,NaN,No Root Cause entered,...,Depot,Craigentinny,2022-04-10,Depot,HB,LNER,Depot,Azuma,NaN,NaN
2,485877220403,2022-04-03,A,MHB1,HITACHI - CLASS 800,MHB1,OQI8,MY,MY05,Class 800/801 Couplers,...,Azuma,Hitachi - Class 800,2022-04-10,Fleet,HB,LNER,Azuma,Azuma,NaN,NaN
3,487346220404,2022-04-04,A,MHB1,HITACHI - CLASS 800,MHB1,OQI8,M7,M703,PASSENGER DOOR,...,Azuma,Hitachi - Class 800,2022-04-11,Fleet,HB,LNER,Azuma,Azuma,NaN,NaN
4,488234220404,2022-04-04,A,MHBA,FLEET - CLASS 91 LOCOMOTIVE,MHBA,OQIJ,MB,NaN,No Root Cause entered,...,IC225,Cl 91 Loco,2022-04-11,Fleet,HB,LNER,IC225,IC225,NaN,NaN


In [3]:
# Checking how many locations are in dataset
unique_stations = delay_df['Incid. Location'].dropna().unique()
print(unique_stations)

<StringArray>
[            'YRK',             'XCN',             'KGX',         'CRCRFTJ',
     'COLTONJ YRK',         'NNG GRA',             'LDS',             'PBO',
       'NCASTLCEN',          'WHRDJN',
 ...
       'HAGTDNSDG',         'LDS DON',         'FERMEPK', 'ARMLJCN APERLYJ',
         'DONCLCJ',         'COLTONJ',        'DBL BLFD',    'GTHS REST402',
         'AAP GDH',   'GTHS MARSHALM']
Length: 403, dtype: str


In [4]:
# List of LNER main stations
lner_stations = [
    'KGX','SVG', 'PBO', 'GTH', 'RET', 'NNG', 'DON', 'BRO', 'HUL', 'SEL', 'LDS', 'WKF', 'YRK', 'NTR', 'DAR', 'DHM', 'NCL', 'MRP', 'ALM', 'BWK', 'RST', 'DUN', 'EDB', 'HYM', 'FKG', 'STG', 'DBL', 'GLA', 'PHC', 'DUK', 'PIT', 'BLA', 'NWM', 'KIN', 'AVM', 'CRB', 'INV', 'KDY', 'INK', 'LEU',
    'DND', 'ARB', 'MON', 'STN', 'ABD', 'MBR', 'TBY', 'SHP','KGH', 'SKI', 'BDS', 'HRS', 'HRG', 'LCN'
]

In [5]:
# Mapping specific LNER stations and printing how many
node_mapping = {station: idx for idx, station in enumerate(lner_stations)}
print(f"{len(lner_stations)} stations found")

54 stations found


In [6]:
# Dropping stations that are NOT LNER specific
def map_to_lner_network(loc_string):
    loc_string = str(loc_string).upper()
    for station in lner_stations:
        if station in loc_string:
            return station
    return "DROP"

delay_df['GNN_Node'] = delay_df['Incid. Location'].apply(map_to_lner_network)

# Dropping and Checking how many rows are kept
gnn_df = delay_df[delay_df['GNN_Node'] != 'DROP'].copy()
print(f"original rows: {len(delay_df)}")
print(f"rows mapped to lner network: {len(gnn_df)}")

original rows: 3925
rows mapped to lner network: 2870


In [7]:
# Mapping thr topology
physical_connections = [
    # MAINLINE
    ('KGX', 'SVG'), #  London King's Cross to Stevenage
    ('SVG', 'PBO'), # Stevenage to Peterborough
    ('PBO', 'GTH'),
    ('GTH', 'NNG'),
    ('NNG', 'RET'),
    ('RET', 'DON'),
    ('DON', 'YRK'),
    ('YRK', 'NTR'),
    ('NTR', 'DAR'),
    ('DAR', 'DHM'),
    ('DHM', 'NCL'),
    ('NCL', 'MRP'),
    ('MRP', 'ALM'),
    ('ALM', 'BWK'),
    ('BWK', 'RST'),
    ('RST', 'DUN'),
    ('DUN', 'EDB'),

    #Inverness
    ('EDB', 'HYM'),
    ('HYM', 'FKG'),
    ('FKG', 'STG'),
    ('STG', 'DBL'),
    ('DBL', 'GLA'),
    ('GLA', 'PHC'),
    ('PHC', 'DUK'),
    ('DUK', 'PIT'),
    ('PIT', 'BLA'),
    ('BLA', 'NWM'),
    ('NWM', 'KIN'),
    ('KIN', 'AVM'),
    ('AVM', 'CRB'),
    ('CRB', 'INV'),

    #Aberdeen
    ('HYM','INK'),
    ('INK', 'KDY'),
    ('KDY', 'LEU'),
    ('LEU', 'DND'),
    ('DND', 'ARB'),
    ('ARB', 'MON'),
    ('MON', 'STN'),
    ('STN', 'ABD'),

    #BRANCHES
    ('NNG', 'LCN'), #newark to lincoln

    ('DON','SEL'), #Doncaster to hull
    ('SEL', 'BRO'),
    ('BRO', 'HUL'),

    ('NTR', 'TBY'), #northallerton to middlesbrough
    ('TBY', 'MBR'),

    ('DON', 'WKF'), #Doncaster to harrogate
    ('WKF', 'LDS'),
    ('LDS', 'HRS'),
    ('HRS', 'HRG'),

    ('LDS', 'SHP'), #Leeds to Skipton
    ('SHP', 'KGH'),
    ('KGH', 'SKI'),

    ('SHP','BDS') # Shipley to Bradford Forster Square

]

In [8]:
# Creating Routes
valid_edges = []
for station_A, station_B in physical_connections:
    if station_A in node_mapping and station_B in node_mapping:
        id_A = node_mapping[station_A]
        id_B = node_mapping[station_B]
        valid_edges.append([id_A, id_B])
        valid_edges.append([id_B, id_A])

edge_index = torch.tensor(valid_edges, dtype=torch.long).t().contiguous()

print(edge_index.shape)

torch.Size([2, 106])


*Creating graphs*


In [9]:
# Importing pandas & plotly to manipulate & visualise data
import pandas as pd
import plotly.express as px

# Loading dataset
delay_df = pd.read_excel('data/Fleet Dealy Data from P2301 to P2611.xlsx')
delay_df['Date'] = pd.to_datetime(delay_df['Date']).dt.date

# Removing outliers
ninety_ninth_percentile = delay_df['Tot. Mins'].quantile(0.99)
delay_df = delay_df[delay_df['Tot. Mins'] <= ninety_ninth_percentile]

print(f"Removed outliers. Max delay is now: {delay_df['Tot. Mins'].max()} minutes.")

# Loading regional weather data
w_london = pd.read_csv('data/metro_weather_london.csv', skiprows=3)
w_york = pd.read_csv('data/metro_weather_york.csv', skiprows=3)
w_newc = pd.read_csv('data/metro_weather_durham.csv', skiprows=3)
w_edin = pd.read_csv('data/metro_weather_perth.csv', skiprows=3)

Removed outliers. Max delay is now: 343.0 minutes.


In [10]:
# Renaming
file_mapping = {
    'London': w_london,
    'York': w_york,
    'Newcastle': w_newc,
    'Edinburgh': w_edin,
}

# Creating master df for weather
weather_dfs = []
for region_name, df in file_mapping.items():
    df['Date'] = pd.to_datetime(df['time']).dt.date
    df['Region'] = region_name
    df = df[['Date', 'Region', 'wind_gusts_10m_max (km/h)', 'precipitation_sum (mm)']]

    df.rename(columns={
        'wind_gusts_10m_max (km/h)':'Wind_Gusts',
        'precipitation_sum (mm)':'Precipitation',
    }, inplace=True)
    weather_dfs.append(df)

all_weather_dfs = pd.concat(weather_dfs)

In [11]:
# Mapping the regions to the stations
region_mapping = {
    'London' : ['KGX', 'SVG', 'PBO'],
    'York' : ['GTH', 'NNG','LCN', 'RET', 'DON', 'YRK', 'NTR'],
    'Newcastle' : ['DAR', 'DHM', 'NCL', 'MRP'],
    'Edinburgh' : ['ALM', 'BWK', 'RST', 'DUN', 'EDB', 'INV']
}
station_to_region = {}
for region, stations in region_mapping.items():
    for station in stations:
        station_to_region[station] = region

delay_df['Region'] = delay_df['Incid. Location'].map(station_to_region)

Creating master df for stations with total delay and weather

In [12]:
delay_df = delay_df.dropna(subset=['Region'])
daily_delays = delay_df.groupby(['Date', 'Region'])['Tot. Mins'].sum().reset_index()

In [13]:
merged_df = pd.merge(daily_delays, all_weather_dfs, on=['Date', 'Region'], how='inner')

In [14]:
# Checking first five rows
merged_df.head()

,Date,Region,Tot. Mins,Wind_Gusts,Precipitation
0,2022-04-01,York,8.0,38.9,4.9
1,2022-04-03,London,6.0,24.8,0.1
2,2022-04-04,London,3.0,47.5,3.4
3,2022-04-05,London,3.0,42.8,0.1
4,2022-04-06,London,8.0,62.3,2.4


In [15]:
# Scatter plot using plotly
fig1 = px.scatter(
    merged_df,
    x='Date',
    y='Tot. Mins',
    size='Wind_Gusts',
    color='Region',
    hover_data='Precipitation',
    title='LNER Fleet Delays vs Weather Events (Bubble Size = Wind Gusts)',
    labels={'Tot. Mins': 'Total Delay Minutes', 'Date': 'Date'},
    size_max=25,
    opacity=0.7
)
fig1.show()

In [16]:
fig2 = px.scatter(
    merged_df,
    x='Precipitation',
    y='Tot. Mins',
    color='Region',
    facet_col='Region', # Creates 4 separate mini-graphs side-by-side
    title='Correlation: Does Heavy Rain Cause More Delay Minutes?',
    labels={'Tot. Mins': 'Delay Mins', 'Precipitation': 'Precipitation (mm)'}
)

fig2.show()

In [17]:
# Creating Heatmap where most delays are
choke_points = delay_df.groupby('Incid. Location')['Tot. Mins'].sum().reset_index()
choke_points = choke_points[choke_points['Tot. Mins'] > 30]

In [18]:
station_coords = {
    'KGX': {'Lat': 51.5320, 'Lon': -0.1240, 'Station_Name': 'London Kings Cross'},
    'SVG': {'Lat': 51.9028, 'Lon': -0.2064, 'Station_Name': 'Stevenage'},
    'PBO': {'Lat': 52.5738, 'Lon': -0.2505, 'Station_Name': 'Peterborough'},
    'GTH': {'Lat': 52.9126, 'Lon': -0.6370, 'Station_Name': 'Grantham'},
    'NNG': {'Lat': 53.0805, 'Lon': -0.8172, 'Station_Name': 'Newark Northgate'},
    'RET': {'Lat': 53.3188, 'Lon': -0.9490, 'Station_Name': 'Retford'},
    'DON': {'Lat': 53.5222, 'Lon': -1.1396, 'Station_Name': 'Doncaster'},
    'YRK': {'Lat': 53.9580, 'Lon': -1.0932, 'Station_Name': 'York'},
    'DAR': {'Lat': 54.5209, 'Lon': -1.5463, 'Station_Name': 'Darlington'},
    'DHM': {'Lat': 54.7795, 'Lon': -1.5816, 'Station_Name': 'Durham'},
    'NCL': {'Lat': 54.9683, 'Lon': -1.6174, 'Station_Name': 'Newcastle'},
    'MRP': {'Lat': 55.1633, 'Lon': -1.6811, 'Station_Name': 'Morpeth'},
    'ALM': {'Lat': 55.3888, 'Lon': -1.6373, 'Station_Name': 'Alnmouth'},
    'BWK': {'Lat': 55.7720, 'Lon': -2.0076, 'Station_Name': 'Berwick-upon-Tweed'},
    'EDB': {'Lat': 55.9520, 'Lon': -3.1890, 'Station_Name': 'Edinburgh Waverley'},
    'INV': {'Lat': 57.4812, 'Lon': -4.2238, 'Station_Name': 'Inverness'},
    'LDS': {'Lat': 53.7952, 'Lon': -1.5476, 'Station_Name': 'Leeds'}
}

In [19]:
coords_df = pd.DataFrame.from_dict(station_coords, orient='index').reset_index()
coords_df.rename(columns={'index': 'Incid. Location'}, inplace=True)

In [20]:
map_df = pd.merge(choke_points, coords_df, on='Incid. Location', how='inner')

In [21]:
fig_map = px.scatter_map(
    map_df,
    lat="Lat",
    lon="Lon",
    hover_name="Station_Name",
    hover_data=["Tot. Mins", "Incid. Location"],
    size="Tot. Mins",
    color="Tot. Mins",
    color_continuous_scale=px.colors.sequential.YlOrRd, # yellow/red
    size_max=35,           # bubble size
    zoom=5,
    title="LNER Network Choke Points~ Total Delay Minutes by Location"
)

# setting map style
fig_map.update_layout(
    mapbox_style="carto-positron",
    margin={"r":0,"t":40,"l":0,"b":0} # Reduces whitespace
)

fig_map.show()

In [22]:
# Import for math calc
import numpy as np


# define rainfall bins & create them using pd.cut from Precipitation column
bins = [-0.1, 0.09, 0.9, 10.0, 30.0, 500.0] # 500 is arbitrary
labels = ['No Rain', 'Very Light Rain', 'Light Rain', 'Moderate Rain', 'Heavy Rain']

merged_df['Rain_Category'] = pd.cut(merged_df['Precipitation'], bins=bins, labels=labels)

# calculating average rainfall per region
rain_impact = merged_df.groupby(['Region', 'Rain_Category'])['Tot. Mins'].mean().reset_index()

# plotting bar chart
fig_multiplier = px.bar(
    rain_impact,
    x='Rain_Category',
    y='Tot. Mins',
    color='Rain_Category',
    facet_col='Region',
    category_orders={
        # Presenting south to north
        'Region': ['London', 'York', 'Newcastle', 'Edinburgh'],
        # Presenting from little to a lot of rain
        'Rain_Category': ['No Rain', 'Very Light Rain', 'Light Rain', 'Moderate Rain', 'Heavy Rain']
    },
    title='Regional Weather Multipliers: Average Daily Delays by Rainfall Intensity',
    labels={
        'Tot. Mins': 'Average Delay Minutes per Day',
        'Rain_Category': 'Rainfall Intensity'
    },
    color_discrete_sequence=px.colors.sequential.Blues # blue gradient
)

# white background, no legend
fig_multiplier.update_layout(
    template='plotly_white',
    showlegend=False,
    font=dict(size=12)
)

# removing 'region' from label
fig_multiplier.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

fig_multiplier.show()

In [23]:
# south to north region order
regions = ['London', 'York', 'Newcastle', 'Edinburgh']

# title
print("Percentage Increase in Delays (No Rain vs. Moderate Rain)\n" + "~"*55)

for region in regions:
    # isolating the data based on cur. region
    region_data = rain_impact[rain_impact['Region'] == region]

    # extracting the exact numbers for no. rain and moderate rain
    no_rain_mins = region_data[region_data['Rain_Category'] == 'No Rain']['Tot. Mins'].values[0]
    mod_rain_mins = region_data[region_data['Rain_Category'] == 'Moderate Rain']['Tot. Mins'].values[0]

    # calculating percentage increase
    percentage_increase = ((mod_rain_mins - no_rain_mins) / no_rain_mins) * 100

    # display the results
    print(f"{region.upper()}:")
    print(f"  No Rain Average:       {no_rain_mins:.1f} mins")
    print(f"  Moderate Rain Average: {mod_rain_mins:.1f} mins")
    print(f"  IMPACT:                +{percentage_increase:.1f}%\n")

Percentage Increase in Delays (No Rain vs. Moderate Rain)
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
LONDON:
  No Rain Average:       29.1 mins
  Moderate Rain Average: 32.0 mins
  IMPACT:                +10.1%

YORK:
  No Rain Average:       26.0 mins
  Moderate Rain Average: 31.3 mins
  IMPACT:                +20.3%

NEWCASTLE:
  No Rain Average:       18.4 mins
  Moderate Rain Average: 20.3 mins
  IMPACT:                +10.1%

EDINBURGH:
  No Rain Average:       21.7 mins
  Moderate Rain Average: 43.3 mins
  IMPACT:                +99.8%

